# Telecom Customer Churn — 04. Data Quality Assessment

**Goal:** Identify data-quality issues, quantify them, assess their impact, and document what needs to be addressed in the cleaning stage.

> This notebook diagnoses problems. It does not modify the raw dataset.

## 1. Imports and load the raw dataset

In [ ]:
from pathlib import Path

import pandas as pd

RAW_DATA_PATH = Path(r"D:\Data Analytics\Project\2) Dataset\1) Raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(RAW_DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

## 2. Missing-value audit

We check both the number and percentage of actual Pandas missing values.

This is necessary but not sufficient because a field such as `TotalCharges` may contain blank strings rather than `NaN`.

In [ ]:
missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing["missing_pct"] = (
    missing["missing_count"] / len(df) * 100
)

missing.sort_values("missing_count", ascending=False)

## 3. Investigate `TotalCharges` separately

First inspect its data type and sample values.

This is important because the column is conceptually numeric but may be stored as an object/string field.

In [ ]:
print("dtype:", df["TotalCharges"].dtype)
print("\nLast values including possible blanks:")
print(df["TotalCharges"].value_counts(dropna=False).tail(10))

## 4. Detect values that cannot be interpreted as numbers

`errors="coerce"` temporarily converts non-numeric values to `NaN`.

We use this only for diagnosis here.

In [ ]:
total_charges_numeric = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

invalid_total_charges_mask = total_charges_numeric.isna()

print(
    "Values that cannot be interpreted as numeric:",
    invalid_total_charges_mask.sum()
)

## 5. Inspect the affected records

We investigate the business context before deciding how to clean these records.

In particular, compare `tenure`, `MonthlyCharges`, `Contract`, and `Churn`.

In [ ]:
df.loc[
    invalid_total_charges_mask,
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]
]

## 6. Duplicate-row audit

We check exact duplicate records.

We do not remove anything yet.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())

## 7. Customer-ID uniqueness audit

The expected grain is one customer per row, so `customerID` should normally be unique.

We check missing IDs, unique IDs, and duplicated IDs separately.

In [ ]:
print("Missing customer IDs:", df["customerID"].isna().sum())
print("Unique customer IDs:", df["customerID"].nunique())
print("Rows:", len(df))
print("Duplicated customer IDs:", df["customerID"].duplicated().sum())

## 8. Inspect duplicate customer IDs

If duplicates exist, inspect the full records before deciding what they mean.

Do not automatically delete them.

In [ ]:
duplicate_customer_mask = df["customerID"].duplicated(keep=False)

df.loc[duplicate_customer_mask].sort_values("customerID")

## 9. Categorical consistency audit

We inspect unique values in the main categorical fields.

This can reveal unexpected labels, spelling differences, or values that need standardization.

In [ ]:
categorical_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "Churn"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].unique())

## 10. Whitespace audit

This checks whether text values contain leading or trailing whitespace.

We count the affected values without changing them.

In [ ]:
for col in categorical_cols:
    values = df[col].dropna().astype(str)
    whitespace_count = (values.str.strip() != values).sum()

    if whitespace_count:
        print(f"{col}: {whitespace_count} value(s) with surrounding whitespace")

## 11. Category cardinality audit

Unexpected numbers of categories can indicate inconsistent labels.

We compare the number of unique values across the main categorical fields.

In [ ]:
for col in categorical_cols:
    print(f"{col}: {df[col].nunique(dropna=False)} unique value(s)")

## 12. Target-variable validation

The churn target is critical because almost every business question depends on its correctness.

We verify that the observed values are expected.

In [ ]:
print(df["Churn"].value_counts(dropna=False))
print("\nUnique values:", df["Churn"].unique())

## 13. Numeric validity checks

We check for impossible negative values.

The raw `TotalCharges` column is converted temporarily for this check because it may currently be stored as text.

In [ ]:
print("Negative tenure:", (df["tenure"] < 0).sum())
print("Negative monthly charges:", (df["MonthlyCharges"] < 0).sum())
print("Negative total charges:", (total_charges_numeric < 0).sum())

## 14. Outlier assessment using IQR

We use the IQR rule to flag unusual `MonthlyCharges` values.

These are **potential outliers**, not automatically invalid records.

In [ ]:
Q1 = df["MonthlyCharges"].quantile(0.25)
Q3 = df["MonthlyCharges"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

monthly_charge_outliers = df[
    (df["MonthlyCharges"] < lower_bound) |
    (df["MonthlyCharges"] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential outliers:", len(monthly_charge_outliers))

## 15. Cross-field business-rule checks

The dataset contains logically related service fields.

For example, customers without phone service are expected to have `MultipleLines = No phone service`, and customers without internet service have service fields represented accordingly.

These are semantic checks, not simple missing-value checks.

In [ ]:
phone_inconsistency = df[
    (df["PhoneService"] == "No") &
    (df["MultipleLines"] != "No phone service")
]

internet_dependent_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

internet_inconsistencies = df[
    (df["InternetService"] == "No") &
    (
        df[internet_dependent_cols]
        .ne("No internet service")
        .any(axis=1)
    )
]

print("Phone-service inconsistencies:", len(phone_inconsistency))
print("Internet-service inconsistencies:", len(internet_inconsistencies))

## 16. Compact quality audit

This cell brings the main checks together so we have one quick quality snapshot.

In [ ]:
print("=== DATA QUALITY SNAPSHOT ===")
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Missing cells:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate customer IDs:", int(df["customerID"].duplicated().sum()))
print("Invalid TotalCharges:", int(invalid_total_charges_mask.sum()))
print("Negative tenure:", int((df["tenure"] < 0).sum()))
print("Negative MonthlyCharges:", int((df["MonthlyCharges"] < 0).sum()))
print("Negative TotalCharges:", int((total_charges_numeric < 0).sum()))
print("Phone-service inconsistencies:", len(phone_inconsistency))
print("Internet-service inconsistencies:", len(internet_inconsistencies))